In [72]:
# ==========================================
# CELL 1: Setup and Functions
# ==========================================
nvars = 5 # number of variables
BR = QQ[",".join("x"+str(i) for i in range(1, nvars+1))+",z"] # polynomial ring in x1, ..., xnvars, z
# BR is a global variable
BR.inject_variables() # make it so you can use those variables
SymmetricFunctions(QQ).inject_shorthands(verbose=False) # define the bases of symmetric functions

def do_P_k(k, n):
    # we are computing p_k[s_n] = s_n[p_k]
    # evaluate at the generators gens = BR.gens()[:-1] = (x1, x2, x3, x4)
    global BR, nvars
    CR = s[1].expand(nvars).parent()
    return s[n].expand(nvars).subs({CR.gens()[i] : BR('x'+str(i+1))**k for i in range(nvars)})

@cached_function
def do_P_lambda(la, n):
    global BR, nvars
    a = BR(expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1))-BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars))))
    return sum(c * mul(BR('x'+str(i+1))^(v[i]-(nvars-i-1)) for i in range(nvars)) \
               for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))

"""
This is the denominator that you are trying to determine
"""
@cached_function
def den_guess():
    global BR
    m1 = z*x1**5
    m2 = z**2*x1**5*x2**5
    m3 = z*x1**4*x2 
    m4 = z*x1**3*x2**2    
    m5 = z*x1**3*x2*x3
    m6 = z*x1**2*x2**2*x3
    m7 = z**2*x1**4*x2**3*x3**3
    m8 = z**3*x1**5*x2**5*x3**5
    m9 = z**4*x1**5*x2**5*x3**5*x4**5
    m10 = z**3*x1**4*x2**4*x3**4*x4**3
    m11 = z**2*x1**3*x2**3*x3**2*x4**2
    m12 = z**2*x1**4*x2**4*x3*x4
    m13 = z*x1**2*x2*x3*x4
    m14 = z**2*x1**3*x2**3*x3**3*x4 
    m15 = z*x1*x2*x3*x4*x5
    return BR((1-m1)*(1-m2)**3*(1-m3)*(1-m4)*(1-m5)**3*(1-m6)**2*(1-m7)*(1-m8)**3*(1-m9)*(1-m10)*(1-m11)**3*(1-m12)*(1-m13)**3*(1-m14)*(1-m15))

# Global cache to expand the denominator exactly once
EXPANDED_DENOMINATOR = None

def get_den_expanded():
    global EXPANDED_DENOMINATOR
    if EXPANDED_DENOMINATOR is None:
        EXPANDED_DENOMINATOR = den_guess()
    return EXPANDED_DENOMINATOR

@cached_function
def den_coeff(d):
    global BR, z
    # Extremely fast native coefficient extraction (bypassing SR completely)
    return get_den_expanded().coefficient({z: d})

def calc_num(la, d):
    return sum(den_coeff(d-r) * do_P_lambda(Partition(la), r) for r in range(d+1))

Defining x1, x2, x3, x4, x5, z


In [30]:
# ==========================================
# CELL 2: Execution Loop
# ==========================================
out = 0

print("Warming up native denominator expansion... (takes a few seconds)")
get_den_expanded()
print("Expansion complete! Starting degrees...\n")

for d in range(0, 55):
    CC = calc_num([1,1,1,1], d)
    if CC:
        CC_list = list(CC)
        if len(CC_list) > 6:
            front = CC_list[:3]
            back = CC_list[-3:]
            print(d, len(CC_list), "FRONT:", front, "BACK:", back)
        else:
            print(d, len(CC_list), CC_list)
    else:
        print(d, 0, "[]")
    
    out += z**d * CC

Warming up native denominator expansion... (takes a few seconds)
Expansion complete! Starting degrees...

0 1 [(1, 1)]
1 2 [(2, x1^3*x2), (1, x1^2*x2*x3)]
2 5 [(1, x1^6*x2^2), (1, x1^5*x2^2*x3), (1, x1^4*x2^3*x3), (1, x1^4*x2^2*x3^2), (2, x1^3*x2^3*x3^2)]
3 6 [(-2, x1^8*x2^3*x3), (-1, x1^7*x2^4*x3), (-2, x1^7*x2^3*x3^2), (-4, x1^6*x2^4*x3^2), (1, x1^6*x2^3*x3^3), (1, x1^5*x2^4*x3^3)]
4 7 FRONT: [(1, x1^10*x2^4*x3^2), (-2, x1^9*x2^5*x3^2), (-1, x1^9*x2^4*x3^3)] BACK: [(-1, x1^7*x2^6*x3^3), (-2, x1^7*x2^5*x3^4), (1, x1^6*x2^6*x3^4)]
5 6 [(1, x1^11*x2^6*x3^3), (1, x1^10*x2^7*x3^3), (-4, x1^10*x2^6*x3^4), (-2, x1^9*x2^7*x3^4), (-1, x1^9*x2^6*x3^5), (-2, x1^8*x2^7*x3^5)]
6 5 [(2, x1^13*x2^7*x3^4), (1, x1^12*x2^8*x3^4), (1, x1^12*x2^7*x3^5), (1, x1^11*x2^8*x3^5), (1, x1^10*x2^8*x3^6)]
7 2 [(1, x1^14*x2^9*x3^5), (2, x1^13*x2^9*x3^6)]
8 1 [(1, x1^16*x2^10*x3^6)]
9 0 []
10 0 []
11 0 []
12 0 []
13 0 []
14 0 []


KeyboardInterrupt: 

In [31]:
factor(out)

x1^16*x2^10*x3^6*z^8 + x1^14*x2^9*x3^5*z^7 + 2*x1^13*x2^9*x3^6*z^7 + 2*x1^13*x2^7*x3^4*z^6 + x1^12*x2^8*x3^4*z^6 + x1^12*x2^7*x3^5*z^6 + x1^11*x2^8*x3^5*z^6 + x1^10*x2^8*x3^6*z^6 + x1^11*x2^6*x3^3*z^5 + x1^10*x2^7*x3^3*z^5 - 4*x1^10*x2^6*x3^4*z^5 - 2*x1^9*x2^7*x3^4*z^5 - x1^9*x2^6*x3^5*z^5 - 2*x1^8*x2^7*x3^5*z^5 + x1^10*x2^4*x3^2*z^4 - 2*x1^9*x2^5*x3^2*z^4 - x1^9*x2^4*x3^3*z^4 - 2*x1^8*x2^5*x3^3*z^4 - x1^7*x2^6*x3^3*z^4 - 2*x1^7*x2^5*x3^4*z^4 + x1^6*x2^6*x3^4*z^4 - 2*x1^8*x2^3*x3*z^3 - x1^7*x2^4*x3*z^3 - 2*x1^7*x2^3*x3^2*z^3 - 4*x1^6*x2^4*x3^2*z^3 + x1^6*x2^3*x3^3*z^3 + x1^5*x2^4*x3^3*z^3 + x1^6*x2^2*z^2 + x1^5*x2^2*x3*z^2 + x1^4*x2^3*x3*z^2 + x1^4*x2^2*x3^2*z^2 + 2*x1^3*x2^3*x3^2*z^2 + 2*x1^3*x2*z + x1^2*x2*x3*z + 1

In [32]:
(x1^16*x2^10*x3^6*z^8 + x1^14*x2^9*x3^5*z^7 + 2*x1^13*x2^9*x3^6*z^7 + 2*x1^13*x2^7*x3^4*z^6 + x1^12*x2^8*x3^4*z^6 + x1^12*x2^7*x3^5*z^6 + x1^11*x2^8*x3^5*z^6 + x1^10*x2^8*x3^6*z^6 + x1^11*x2^6*x3^3*z^5 + x1^10*x2^7*x3^3*z^5 - 4*x1^10*x2^6*x3^4*z^5 - 2*x1^9*x2^7*x3^4*z^5 - x1^9*x2^6*x3^5*z^5 - 2*x1^8*x2^7*x3^5*z^5 + x1^10*x2^4*x3^2*z^4 - 2*x1^9*x2^5*x3^2*z^4 - x1^9*x2^4*x3^3*z^4 - 2*x1^8*x2^5*x3^3*z^4 - x1^7*x2^6*x3^3*z^4 - 2*x1^7*x2^5*x3^4*z^4 + x1^6*x2^6*x3^4*z^4 - 2*x1^8*x2^3*x3*z^3 - x1^7*x2^4*x3*z^3 - 2*x1^7*x2^3*x3^2*z^3 - 4*x1^6*x2^4*x3^2*z^3 + x1^6*x2^3*x3^3*z^3 + x1^5*x2^4*x3^3*z^3 + x1^6*x2^2*z^2 + x1^5*x2^2*x3*z^2 + x1^4*x2^3*x3*z^2 + x1^4*x2^2*x3^2*z^2 + 2*x1^3*x2^3*x3^2*z^2 + 2*x1^3*x2*z + x1^2*x2*x3*z + 1) == x1^16*x2^10*x3^6*z^8 + x1^14*x2^9*x3^5*z^7 + 2*x1^13*x2^9*x3^6*z^7 + 2*x1^13*x2^7*x3^4*z^6 + x1^12*x2^8*x3^4*z^6 + x1^12*x2^7*x3^5*z^6 + x1^11*x2^8*x3^5*z^6 + x1^10*x2^8*x3^6*z^6 + x1^11*x2^6*x3^3*z^5 + x1^10*x2^7*x3^3*z^5 - 4*x1^10*x2^6*x3^4*z^5 - 2*x1^9*x2^7*x3^4*z^5 - x1^9*x2^6*x3^5*z^5 - 2*x1^8*x2^7*x3^5*z^5 + x1^10*x2^4*x3^2*z^4 - 2*x1^9*x2^5*x3^2*z^4 - x1^9*x2^4*x3^3*z^4 - 2*x1^8*x2^5*x3^3*z^4 - x1^7*x2^6*x3^3*z^4 - 2*x1^7*x2^5*x3^4*z^4 + x1^6*x2^6*x3^4*z^4 - 2*x1^8*x2^3*x3*z^3 - x1^7*x2^4*x3*z^3 - 2*x1^7*x2^3*x3^2*z^3 - 4*x1^6*x2^4*x3^2*z^3 + x1^6*x2^3*x3^3*z^3 + x1^5*x2^4*x3^3*z^3 + x1^6*x2^2*z^2 + x1^5*x2^2*x3*z^2 + x1^4*x2^3*x3*z^2 + x1^4*x2^2*x3^2*z^2 + 2*x1^3*x2^3*x3^2*z^2 + 2*x1^3*x2*z + x1^2*x2*x3*z + 1


True

P2_2var: correct
P3_3var: correct
P4_3var: correct 
P4_4var: correct 
P5_5var: correct
P11_2var: correct
P21_3var: correct
P22_4var: correct
P31_4var: correct
P32_5var: correct
P41_5var: ~4 days to compute numerator
P111_3var: correct
P211_4var: correct
P1111_4var: correct
P2111_5var: 1 to 3 days to compute numerator
P221_5var: 3 days to 2 weeks to compute numerator
P11111_5var: 1 week to 2 weeks to compute numerator
P311_5var: Unknown since I still need to compute P311_4var

In [9]:
out/den_guess()

(-x1^34*x2^21*x3^5*z^12 + 2*x1^33*x2^22*x3^5*z^12 - x1^31*x2^24*x3^5*z^12 - 2*x1^33*x2^21*x3^6*z^12 - 5*x1^32*x2^22*x3^6*z^12 - 9*x1^31*x2^23*x3^6*z^12 - x1^30*x2^24*x3^6*z^12 + 3*x1^33*x2^20*x3^7*z^12 + 2*x1^32*x2^21*x3^7*z^12 - 5*x1^31*x2^22*x3^7*z^12 + 4*x1^29*x2^24*x3^7*z^12 + 2*x1^28*x2^25*x3^7*z^12 + 3*x1^32*x2^20*x3^8*z^12 - 5*x1^31*x2^21*x3^8*z^12 - 7*x1^30*x2^22*x3^8*z^12 + x1^28*x2^24*x3^8*z^12 + x1^27*x2^25*x3^8*z^12 - x1^33*x2^18*x3^9*z^12 - 7*x1^32*x2^19*x3^9*z^12 + 5*x1^31*x2^20*x3^9*z^12 + 5*x1^30*x2^21*x3^9*z^12 + x1^29*x2^22*x3^9*z^12 - 3*x1^28*x2^23*x3^9*z^12 - 6*x1^27*x2^24*x3^9*z^12 - 2*x1^26*x2^25*x3^9*z^12 - 3*x1^32*x2^18*x3^10*z^12 - 12*x1^31*x2^19*x3^10*z^12 - 5*x1^30*x2^20*x3^10*z^12 + 3*x1^29*x2^21*x3^10*z^12 + 15*x1^28*x2^22*x3^10*z^12 + 11*x1^27*x2^23*x3^10*z^12 - x1^26*x2^24*x3^10*z^12 - x1^25*x2^25*x3^10*z^12 + 2*x1^32*x2^17*x3^11*z^12 + 5*x1^31*x2^18*x3^11*z^12 - 2*x1^30*x2^19*x3^11*z^12 - 12*x1^29*x2^20*x3^11*z^12 - 4*x1^28*x2^21*x3^11*z^12 + 5*x1^26*x2^

Execution Loop that tracks ETA

In [73]:
# ==========================================
# CELL 2: Execution Loop with ETA
# ==========================================
import time
import numpy as np

out = 0
target_degree = 45 # Matches the upper bound of your loop
time_history = []
degree_history = []

print("Warming up native denominator expansion... (takes a few seconds)")
get_den_expanded()
print("Expansion complete! Starting degrees...\n")

for d in range(0, target_degree):
    start_time = time.time()
    
    # Calculate the numerator coefficient for the current degree
    CC = calc_num([1,1,1,1,1], d)
    
    elapsed = time.time() - start_time
    time_history.append(elapsed)
    degree_history.append(d)
    
    if d < target_degree and len(time_history) >= 8:
        try:
            # Filter out early degrees to avoid warm-up noise
            mask = np.array(degree_history) > 6
            x_data = np.array(degree_history)[mask]
            y_data = np.array(time_history)[mask]
            
            # Fit Log-Quadratic: log(time) = A*d^2 + B*d + C
            # This perfectly models a multiplier that decays by a constant factor!
            coeffs = np.polyfit(x_data, np.log(y_data), deg=2)
            A, B, C = coeffs[0], coeffs[1], coeffs[2]
            
            # Predict remaining times using the extrapolated parabola
            remaining_degrees = np.arange(d + 1, target_degree)
            predicted_times = np.exp(A * (remaining_degrees**2) + B * remaining_degrees + C)
            
            projected_remaining = np.sum(predicted_times)
            
            # The decay factor (e.g. your ~0.967) is exactly e^(2A)
            decay_factor = np.exp(2 * A)
            
            eta_str = f" | ETA to d={target_degree}: ~{projected_remaining/60:.1f} min (Decay: {decay_factor:.3f})"
        except Exception:
            eta_str = ""
    else:
        eta_str = ""

    if CC:
        CC_list = list(CC)
        if len(CC_list) > 6:
            front = CC_list[:3]
            back = CC_list[-3:]
            print(f"{d:02d} {len(CC_list):4d} FRONT: {front} BACK: {back} | Time: {elapsed:.2f}s{eta_str}")
        else:
            print(f"{d:02d} {len(CC_list):4d} {CC_list} | Time: {elapsed:.2f}s{eta_str}")
    else:
        print(f"{d:02d}    0 [] | Time: {elapsed:.2f}s{eta_str}")
    
    out += z**d * CC

Warming up native denominator expansion... (takes a few seconds)
Expansion complete! Starting degrees...

00    1 [(1, 1)] | Time: 0.01s
01    5 [(3, x1^4*x2), (4, x1^3*x2^2), (3, x1^3*x2*x3), (3, x1^2*x2^2*x3), (1, x1^2*x2*x3*x4)] | Time: 0.02s
02   18 FRONT: [(1, x1^8*x2^2), (7, x1^7*x2^3), (10, x1^6*x2^4)] BACK: [(4, x1^3*x2^3*x3^3*x4), (1, x1^4*x2^2*x3^2*x4^2), (3, x1^3*x2^3*x3^2*x4^2)] | Time: 0.05s
03   42 FRONT: [(-1, x1^11*x2^4), (5, x1^10*x2^5), (4, x1^9*x2^6)] BACK: [(1, x1^6*x2^3*x3^3*x4^3), (-1, x1^5*x2^4*x3^3*x4^3), (3, x1^4*x2^4*x3^4*x4^3)] | Time: 0.11s
04   78 FRONT: [(-2, x1^13*x2^7), (-4, x1^12*x2^8), (-5, x1^11*x2^9)] BACK: [(1, x1^8*x2^4*x3^4*x4^4), (5, x1^6*x2^6*x3^4*x4^4), (2, x1^6*x2^5*x3^5*x4^4)] | Time: 0.25s
05  133 FRONT: [(-3, x1^16*x2^9), (-10, x1^15*x2^10), (-7, x1^14*x2^11)] BACK: [(-4, x1^9*x2^6*x3^5*x4^5), (-8, x1^8*x2^7*x3^5*x4^5), (-4, x1^8*x2^6*x3^6*x4^5)] | Time: 0.53s
06  198 FRONT: [(-4, x1^18*x2^12), (-3, x1^17*x2^13), (9, x1^19*x2^10*x3)] BACK: 

/tmp/ipykernel_317622/509212161.py:35: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(x_data, np.log(y_data), deg=Integer(2))


07  281 FRONT: [(-1, x1^21*x2^14), (12, x1^21*x2^13*x3), (13, x1^20*x2^14*x3)] BACK: [(9, x1^11*x2^9*x3^8*x4^7), (-8, x1^10*x2^10*x3^8*x4^7), (-3, x1^10*x2^9*x3^9*x4^7)] | Time: 1.89s | ETA to d=45: ~1038.0 min (Decay: 1.009)


/tmp/ipykernel_317622/509212161.py:35: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(x_data, np.log(y_data), deg=Integer(2))


08  378 FRONT: [(3, x1^24*x2^15*x3), (2, x1^23*x2^16*x3), (-1, x1^22*x2^17*x3)] BACK: [(17, x1^12*x2^11*x3^9*x4^8), (3, x1^12*x2^10*x3^10*x4^8), (-3, x1^11*x2^11*x3^10*x4^8)] | Time: 3.57s | ETA to d=45: ~41982203039795742146560654311424.0 min (Decay: 1.082)
09  483 FRONT: [(-3, x1^27*x2^16*x3^2), (-6, x1^26*x2^17*x3^2), (-4, x1^25*x2^18*x3^2)] BACK: [(4, x1^13*x2^13*x3^10*x4^9), (-1, x1^14*x2^11*x3^11*x4^9), (9, x1^13*x2^12*x3^11*x4^9)] | Time: 6.41s | ETA to d=45: ~24.8 min (Decay: 0.950)
10  603 FRONT: [(1, x1^30*x2^17*x3^3), (6, x1^29*x2^18*x3^3), (8, x1^28*x2^19*x3^3)] BACK: [(-3, x1^15*x2^14*x3^11*x4^10), (-9, x1^15*x2^13*x3^12*x4^10), (3, x1^14*x2^14*x3^12*x4^10)] | Time: 11.47s | ETA to d=45: ~724.9 min (Decay: 0.972)
11  718 FRONT: [(-2, x1^32*x2^19*x3^4), (5, x1^30*x2^21*x3^4), (-3, x1^29*x2^22*x3^4)] BACK: [(1, x1^16*x2^16*x3^12*x4^11), (3, x1^17*x2^14*x3^13*x4^11), (-9, x1^16*x2^15*x3^13*x4^11)] | Time: 20.15s | ETA to d=45: ~4465.9 min (Decay: 0.978)
12  846 FRONT: [(-3, x

KeyboardInterrupt: 

In [76]:
[404.51/259.52, 259.52/162.39,162.39/99.26,99.26/59.38,59.38/35.00,35.00/20.15]

[1.55868526510481,
 1.59812796354455,
 1.63600644771308,
 1.67160660154934,
 1.69657142857143,
 1.73697270471464]

In [78]:
[1.55868526510481/1.59812796354455, 1.59812796354455/1.63600644771308,1.63600644771308/1.67160660154934,1.72510264489640/1.78203164880041,1.66867769967344/1.72510264489640]

[0.975319436653709,
 0.976846983566917,
 0.978703031082036,
 0.968053876067615,
 0.967291833103445]

In [81]:
total_time = 0
time = 404 
increase = 1.55
for i in range(1,25):
    time = time*increase
    total_time = total_time + time
    increase = increase*0.98
print(total_time/60/60/24)
    

7.49244936610582


In [ ]:
1.4 days. Great.

In [ ]:
22281652504003439123993443243200536813169242413604197979859530875317829057631113616848926274896087107192657470667218147009430597105415618560.0/60/24

In [7]:
# 1. Setup Environment
Sym = SymmetricFunctions(QQ)
Sym.inject_shorthands(verbose=False)
R = PolynomialRing(QQ, 'a, b, x1, x2, x3, x4, x5, z').fraction_field()
R.inject_variables()
x = R.gens()[2:-1]
a = R.gens()[0]
b = R.gens()[1]
z = R.gens()[-1]

# 2. Define Core Functions
def normalize_rational_function(Q):
    S = PolynomialRing(PolynomialRing(QQ, 'a,b'), 'x1,x2,x3,x4,x5,z')
    K = R.fraction_field()
    den = []
    factors = Q.denominator().factor()
    scalar = factors.unit()
    for (factor, exp) in factors:
        c = S(factor).constant_coefficient()
        den.append((K(factor) / c, exp))
        scalar *= c**exp
    den = Factorization(den)
    num = Q.numerator() / scalar
    return (num, den)

def CT(f, g):
    Q = f.subs({z: z / (a * b)}) * g.subs({z: a * b})
    num, den = normalize_rational_function(Q)
    PTa = MacMahonOmega(a, num, den)
    CTa = prod(PTa).subs(b=0)
    return CTa

def P(k):
    return R.one() / R.prod((1 - z * xi**k) for xi in x)

# 3. The Step-by-Step Execution
print("Computing P(1,1)...")
F_11 = CT(P(1), P(1))

print("Computing P(1,1,1)...")
F_111 = CT(P(1), F_11)

print("Computing P(1,1,1,1)...")
F_1111 = CT(P(1), F_111)

print("Computing P(2,1,1,1) [Final Step]...")
F_2111 = CT(P(2), F_1111)

# 4. Extract the Exact Denominator
print("Normalizing final rational function...")
final_num, final_den = normalize_rational_function(F_2111)

print("\n--- EXACT SYMBOLIC DENOMINATOR ---")
print(final_den)

Defining a, b, x1, x2, x3, x4, x5, z
Computing P(1,1)...


KeyboardInterrupt: 